# Teste do `limpeza.py` (Issue #2)

Notebook para validar manualmente as 4 funcoes de `src/limpeza.py` usando o dataset real em `data/raw/`.

Ordem de execucao: `remover_nulos_criticos` -> `corrigir_tipos` -> `validar_dominios` -> `detectar_outliers_amount`.

In [3]:
import os
import sys

# Adiciona o diretório raiz do projeto ao path do Python
sys.path.append(os.path.abspath(os.path.join('..')))


In [4]:
import pandas as pd
from pathlib import Path

from src.limpeza import (
    remover_nulos_criticos,
    corrigir_tipos,
    validar_dominios,
    detectar_outliers_amount,
)

## 1. Carregar o CSV de `data/raw/`

In [5]:
# O ".." sobe uma pasta (sai de 'notebooks' e vai para a raiz do projeto)
pasta_raw = Path("../data/raw")
arquivos_csv = list(pasta_raw.glob("*.csv"))
print("CSVs encontrados:", arquivos_csv)
if arquivos_csv:
    caminho = arquivos_csv[0]
    df = pd.read_csv(caminho)
    display(df.head()) # display() é melhor para renderizar tabelas no Jupyter
else:
    print("Nenhum arquivo CSV encontrado em data/raw/. Verifique se o dataset está lá!")

CSVs encontrados: [WindowsPath('../data/raw/synthetic_fraud_dataset.csv')]


,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud
0,9608,363,4922.587542,ATM,Travel,TR,12,0.992347,0.947908,1
1,456,692,48.018303,QR,Food,US,21,0.168571,0.224057,0
2,4747,587,136.881960,Online,Travel,TR,14,0.296127,0.125058,0
3,6934,445,80.534719,POS,Clothing,TR,23,0.124801,0.159243,0
4,1646,729,120.041158,Online,Grocery,FR,16,0.098129,0.027542,0


## 2. Inspecionar antes de rodar qualquer funcao

Confirma nomes de colunas e tipos antes de aplicar o pipeline (evita `KeyError`).

In [6]:
print("Linhas:", len(df))
print("Colunas:", list(df.columns))
print("\nTipos:")
print(df.dtypes)
print("\nNulos por coluna:")
print(df.isna().sum())

Linhas: 10000
Colunas: ['transaction_id', 'user_id', 'amount', 'transaction_type', 'merchant_category', 'country', 'hour', 'device_risk_score', 'ip_risk_score', 'is_fraud']

Tipos:
transaction_id         int64
user_id                int64
amount               float64
transaction_type         str
merchant_category        str
country                  str
hour                   int64
device_risk_score    float64
ip_risk_score        float64
is_fraud               int64
dtype: object

Nulos por coluna:
transaction_id       0
user_id              0
amount               0
transaction_type     0
merchant_category    0
country              0
hour                 0
device_risk_score    0
ip_risk_score        0
is_fraud             0
dtype: int64


## 3. Etapa 1 - `remover_nulos_criticos`

In [7]:
df_limpo = remover_nulos_criticos(df)
print(f"Linhas: {len(df)} -> {len(df_limpo)}")
df_limpo.head()

Linhas: 10000 -> 10000


,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud
0,9608,363,4922.587542,ATM,Travel,TR,12,0.992347,0.947908,1
1,456,692,48.018303,QR,Food,US,21,0.168571,0.224057,0
2,4747,587,136.881960,Online,Travel,TR,14,0.296127,0.125058,0
3,6934,445,80.534719,POS,Clothing,TR,23,0.124801,0.159243,0
4,1646,729,120.041158,Online,Grocery,FR,16,0.098129,0.027542,0


## 4. Etapa 2 - `corrigir_tipos`

In [9]:
df_tipado = corrigir_tipos(df_limpo)
print(df_tipado.dtypes)
df_tipado.head()

transaction_id         int64
user_id                int64
amount               float64
transaction_type         str
merchant_category        str
country                  str
hour                   int64
device_risk_score    float64
ip_risk_score        float64
is_fraud               int64
dtype: object


,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud
0,9608,363,4922.587542,ATM,Travel,TR,12,0.992347,0.947908,1
1,456,692,48.018303,QR,Food,US,21,0.168571,0.224057,0
2,4747,587,136.881960,Online,Travel,TR,14,0.296127,0.125058,0
3,6934,445,80.534719,POS,Clothing,TR,23,0.124801,0.159243,0
4,1646,729,120.041158,Online,Grocery,FR,16,0.098129,0.027542,0


## 5. Etapa 3 - `validar_dominios`

Por padrao, valores fora do dominio ficam como `NaN` (nao sao removidos).

In [10]:
df_validado = validar_dominios(df_tipado)
print("Nulos criados em transaction_type / is_fraud:")
print(df_validado[["transaction_type", "is_fraud"]].isna().sum())
df_validado.head()

Nulos criados em transaction_type / is_fraud:
transaction_type    0
is_fraud            0
dtype: int64


,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud
0,9608,363,4922.587542,ATM,Travel,TR,12,0.992347,0.947908,1.0
1,456,692,48.018303,QR,Food,US,21,0.168571,0.224057,0.0
2,4747,587,136.881960,Online,Travel,TR,14,0.296127,0.125058,0.0
3,6934,445,80.534719,POS,Clothing,TR,23,0.124801,0.159243,0.0
4,1646,729,120.041158,Online,Grocery,FR,16,0.098129,0.027542,0.0


## 6. Etapa 4 - `detectar_outliers_amount`

Por padrao, outliers ficam marcados na coluna `is_outlier` (nao sao removidos).

In [12]:
df_final = detectar_outliers_amount(df_validado)
print("Outliers detectados:", df_final["is_outlier"].sum())
df_final.head(10)

Outliers detectados: 187


,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud,is_outlier
0,9608,363,4922.587542,ATM,Travel,TR,12,0.992347,0.947908,1.0,True
1,456,692,48.018303,QR,Food,US,21,0.168571,0.224057,0.0,False
2,4747,587,136.881960,Online,Travel,TR,14,0.296127,0.125058,0.0,False
3,6934,445,80.534719,POS,Clothing,TR,23,0.124801,0.159243,0.0,False
4,1646,729,120.041158,Online,Grocery,FR,16,0.098129,0.027542,0.0,False
5,2183,944,97.108625,POS,Clothing,DE,17,0.235399,0.105454,0.0,False
6,1919,829,166.209262,Online,Travel,UK,12,0.115906,0.223718,0.0,False
7,3479,845,96.512637,Online,Grocery,US,7,0.082250,0.034023,0.0,False
8,6796,129,83.338701,QR,Food,DE,16,0.021774,0.279598,0.0,False
9,5129,249,89.695731,QR,Grocery,UK,6,0.095353,0.136336,0.0,False


## 7. Resumo final

In [13]:
print("Linhas originais:", len(df))
print("Linhas finais:", len(df_final))
print("Colunas finais:", list(df_final.columns))
df_final.describe(include="all")

Linhas originais: 10000
Linhas finais: 10000
Colunas finais: ['transaction_id', 'user_id', 'amount', 'transaction_type', 'merchant_category', 'country', 'hour', 'device_risk_score', 'ip_risk_score', 'is_fraud', 'is_outlier']


,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud,is_outlier
count,10000.00000,10000.000000,10000.000000,10000,10000,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000
unique,NaN,NaN,NaN,4,5,6,NaN,NaN,NaN,NaN,2
top,NaN,NaN,NaN,POS,Food,US,NaN,NaN,NaN,NaN,False
freq,NaN,NaN,NaN,2568,2023,2050,NaN,NaN,NaN,NaN,9813
mean,4999.50000,500.058700,178.142763,NaN,NaN,NaN,14.247100,0.183773,0.184669,0.050000,NaN
std,2886.89568,288.328495,531.647950,NaN,NaN,NaN,5.347383,0.177381,0.175772,0.217956,NaN
min,0.00000,0.000000,1.000000,NaN,NaN,NaN,0.000000,0.000030,0.000009,0.000000,NaN
25%,2499.75000,247.000000,65.084753,NaN,NaN,NaN,10.000000,0.075721,0.077762,0.000000,NaN
50%,4999.50000,503.000000,101.686510,NaN,NaN,NaN,14.000000,0.156583,0.158290,0.000000,NaN
75%,7499.25000,750.250000,138.280872,NaN,NaN,NaN,19.000000,0.234939,0.236968,0.000000,NaN
